# 09 - Domain-Adversarial Neural Network (DANN)

This notebook prepares the DANN experiment used in the normalisation and batch correction chapter.

The goal is to test whether a domain-adversarial model can learn features that are useful for **chemoresistance classification** while being less informative about the **acquisition session / domain**.

The validation is performed at the **file level**, not at the cell level. Complete microscopy files are held out for testing, so cells from the same file are never shared between train and test sets.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import random
import json
import math
from typing import List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# -----------------------------
# Paths
# -----------------------------
PROJECT_ROOT = Path(".")  # change if needed
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "dann"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Experiment settings
# -----------------------------
TIMEPOINT = 14
BATCH_SIZE = 256
LR = 5e-4
N_EPOCHS = 20
NUM_WORKERS = 4

# Image settings
N_CHANNELS = 7
IMG_SIZE = 512
EPS = 1e-8

## 2. File manifest

This manifest should match the processed crops available on disk.

Expected files per sample:

- `crops_path`: NumPy array with shape `(N, 7, 512, 512)`
- `metadata_path`: CSV table with at least `time`, `tile`, `cell_id`, and `group` or equivalent fields

Adjust the paths to match your current repository.

In [ ]:
FILES = [
    {
        "file_label": "CNTL-MB231",
        "group": "Control",
        "group_label": 0,
        "session": 1,
        "domain_label": 0,
        "crops_path": DATA_DIR / "CNTL-MB231_crops.npy",
        "metadata_path": DATA_DIR / "CNTL-MB231_metadata.csv",
    },
    {
        "file_label": "TAMO-MB231",
        "group": "Chemoresistant",
        "group_label": 1,
        "session": 1,
        "domain_label": 0,
        "crops_path": DATA_DIR / "TAMO-MB231_crops.npy",
        "metadata_path": DATA_DIR / "TAMO-MB231_metadata.csv",
    },
    {
        "file_label": "CNTL_75uM_p1",
        "group": "Control",
        "group_label": 0,
        "session": 2,
        "domain_label": 1,
        "crops_path": DATA_DIR / "CNTL_75uM_p1_crops.npy",
        "metadata_path": DATA_DIR / "CNTL_75uM_p1_metadata.csv",
    },
    {
        "file_label": "CNTL_75uM_p2",
        "group": "Control",
        "group_label": 0,
        "session": 2,
        "domain_label": 1,
        "crops_path": DATA_DIR / "CNTL_75uM_p2_crops.npy",
        "metadata_path": DATA_DIR / "CNTL_75uM_p2_metadata.csv",
    },
    {
        "file_label": "CNTL_75uM_p3",
        "group": "Control",
        "group_label": 0,
        "session": 2,
        "domain_label": 1,
        "crops_path": DATA_DIR / "CNTL_75uM_p3_crops.npy",
        "metadata_path": DATA_DIR / "CNTL_75uM_p3_metadata.csv",
    },
    {
        "file_label": "CNTL_75uM_p4",
        "group": "Control",
        "group_label": 0,
        "session": 2,
        "domain_label": 1,
        "crops_path": DATA_DIR / "CNTL_75uM_p4_crops.npy",
        "metadata_path": DATA_DIR / "CNTL_75uM_p4_metadata.csv",
    },
    {
        "file_label": "TAMO_p1",
        "group": "Chemoresistant",
        "group_label": 1,
        "session": 3,
        "domain_label": 2,
        "crops_path": DATA_DIR / "TAMO_p1_crops.npy",
        "metadata_path": DATA_DIR / "TAMO_p1_metadata.csv",
    },
    {
        "file_label": "TAMO_p2",
        "group": "Chemoresistant",
        "group_label": 1,
        "session": 3,
        "domain_label": 2,
        "crops_path": DATA_DIR / "TAMO_p2_crops.npy",
        "metadata_path": DATA_DIR / "TAMO_p2_metadata.csv",
    },
]

manifest = pd.DataFrame(FILES)
manifest

## 3. File-level validation scheme

Each fold holds out complete microscopy files for testing. The held-out set should contain both classes whenever possible.

In [ ]:
FOLDS = [
    {
        "fold": 1,
        "test_files": ["CNTL-MB231", "TAMO-MB231"],
    },
    {
        "fold": 2,
        "test_files": ["CNTL_75uM_p1", "CNTL_75uM_p2", "TAMO_p1"],
    },
    {
        "fold": 3,
        "test_files": ["CNTL_75uM_p3", "CNTL_75uM_p4", "TAMO_p2"],
    },
]

all_files = set(manifest["file_label"])

for fold in FOLDS:
    fold["train_files"] = sorted(list(all_files - set(fold["test_files"])))

pd.DataFrame(FOLDS)

## 4. Dataset utilities

The dataset filters each file to a single timepoint and returns:

- image crop: `(7, 512, 512)`
- class label: `0 = Control`, `1 = Chemoresistant`
- domain label: acquisition domain/session
- file label

If your metadata uses another name for the timepoint column, update `TIME_COL`.

In [ ]:
TIME_COL = "time"  # change to "timepoint" if needed

class ChemoresistanceCropsDataset(Dataset):
    def __init__(self, files: List[dict], timepoint: int, transform=None, preload: bool = False):
        self.files = files
        self.timepoint = timepoint
        self.transform = transform
        self.preload = preload
        self.index = []
        self.arrays_cache = {}

        for f in files:
            crops_path = Path(f["crops_path"])
            metadata_path = Path(f["metadata_path"])

            if not crops_path.exists():
                raise FileNotFoundError(f"Missing crops file: {crops_path}")
            if not metadata_path.exists():
                raise FileNotFoundError(f"Missing metadata file: {metadata_path}")

            meta = pd.read_csv(metadata_path)
            if TIME_COL not in meta.columns:
                raise ValueError(
                    f"Column '{TIME_COL}' not found in {metadata_path}. "
                    f"Available columns: {list(meta.columns)}"
                )

            time_idx = meta.index[meta[TIME_COL] == timepoint].to_numpy()
            if len(time_idx) == 0:
                print(f"Warning: no cells for {f['file_label']} at timepoint {timepoint}")
                continue

            if preload:
                self.arrays_cache[f["file_label"]] = np.load(crops_path, mmap_mode=None)

            for idx in time_idx:
                self.index.append({
                    "file_label": f["file_label"],
                    "array_idx": int(idx),
                    "crops_path": crops_path,
                    "class_label": int(f["group_label"]),
                    "domain_label": int(f["domain_label"]),
                    "group": f["group"],
                    "session": int(f["session"]),
                })

        if len(self.index) == 0:
            raise ValueError("Dataset is empty. Check timepoint and paths.")

    def __len__(self):
        return len(self.index)

    def _load_array(self, item):
        file_label = item["file_label"]
        if self.preload and file_label in self.arrays_cache:
            arr = self.arrays_cache[file_label]
        else:
            arr = np.load(item["crops_path"], mmap_mode="r")
        return arr[item["array_idx"]]

    def __getitem__(self, idx):
        item = self.index[idx]
        x = self._load_array(item).astype(np.float32)

        if x.shape[0] != N_CHANNELS:
            raise ValueError(f"Expected {N_CHANNELS} channels, got shape {x.shape}")

        if self.transform is not None:
            x = self.transform(x)

        x = torch.from_numpy(x).float()
        y = torch.tensor(item["class_label"], dtype=torch.long)
        d = torch.tensor(item["domain_label"], dtype=torch.long)

        return {
            "image": x,
            "class_label": y,
            "domain_label": d,
            "file_label": item["file_label"],
            "group": item["group"],
            "session": item["session"],
        }


def build_dataset(file_labels: List[str], timepoint: int, preload: bool = False):
    selected = [f for f in FILES if f["file_label"] in file_labels]
    return ChemoresistanceCropsDataset(selected, timepoint=timepoint, preload=preload)


def summarise_dataset(ds: Dataset):
    rows = []
    for item in ds.index:
        rows.append({
            "file_label": item["file_label"],
            "group": item["group"],
            "class_label": item["class_label"],
            "domain_label": item["domain_label"],
            "session": item["session"],
        })
    return pd.DataFrame(rows).groupby(
        ["file_label", "group", "class_label", "domain_label", "session"]
    ).size().reset_index(name="n_cells")

## 5. DANN model

The model has:

1. A convolutional feature extractor.
2. A class head for Control vs Chemoresistant prediction.
3. A domain head for acquisition-domain prediction.
4. A gradient reversal layer before the domain head.

In [ ]:
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


class GradientReversal(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, lambda_=1.0):
        return GradientReversalFunction.apply(x, lambda_)


class DANN(nn.Module):
    def __init__(self, n_channels=7, n_classes=2, n_domains=3, embedding_dim=256):
        super().__init__()

        self.feature_extractor = nn.Sequential(
            nn.Conv2d(n_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
        )

        self.class_head = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, n_classes),
        )

        self.domain_head = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, n_domains),
        )

        self.grl = GradientReversal()

    def forward(self, x, lambda_=1.0):
        z = self.feature_extractor(x)
        z = self.embedding(z)
        class_logits = self.class_head(z)
        z_rev = self.grl(z, lambda_)
        domain_logits = self.domain_head(z_rev)
        return class_logits, domain_logits, z

## 6. Training and evaluation utilities

In [ ]:
def dann_lambda(epoch: int, n_epochs: int):
    """
    Standard DANN schedule:
    lambda = 2 / (1 + exp(-10 * p)) - 1,
    where p goes from 0 to 1 across training.
    """
    p = epoch / max(n_epochs - 1, 1)
    return 2.0 / (1.0 + math.exp(-10 * p)) - 1.0


def make_weighted_sampler(ds: ChemoresistanceCropsDataset):
    labels = np.array([item["class_label"] for item in ds.index])
    class_counts = np.bincount(labels, minlength=2)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True,
    )


def train_one_epoch(model, loader, optimizer, epoch, n_epochs, domain_weight=1.0):
    model.train()

    total_loss = 0.0
    total_class_loss = 0.0
    total_domain_loss = 0.0
    n = 0

    lambda_ = dann_lambda(epoch, n_epochs)

    for batch in loader:
        x = batch["image"].to(DEVICE)
        y = batch["class_label"].to(DEVICE)
        d = batch["domain_label"].to(DEVICE)

        optimizer.zero_grad()

        class_logits, domain_logits, _ = model(x, lambda_=lambda_)
        class_loss = F.cross_entropy(class_logits, y)
        domain_loss = F.cross_entropy(domain_logits, d)

        loss = class_loss + domain_weight * domain_loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_class_loss += class_loss.item() * bs
        total_domain_loss += domain_loss.item() * bs
        n += bs

    return {
        "loss": total_loss / n,
        "class_loss": total_class_loss / n,
        "domain_loss": total_domain_loss / n,
        "lambda": lambda_,
    }


@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    y_true, y_pred, y_prob = [], [], []
    d_true, d_pred = [], []
    file_labels = []

    for batch in loader:
        x = batch["image"].to(DEVICE)
        y = batch["class_label"].cpu().numpy()
        d = batch["domain_label"].cpu().numpy()

        class_logits, domain_logits, _ = model(x, lambda_=0.0)

        probs = torch.softmax(class_logits, dim=1)[:, 1].detach().cpu().numpy()
        preds = torch.argmax(class_logits, dim=1).detach().cpu().numpy()
        dom_preds = torch.argmax(domain_logits, dim=1).detach().cpu().numpy()

        y_true.extend(y.tolist())
        y_pred.extend(preds.tolist())
        y_prob.extend(probs.tolist())

        d_true.extend(d.tolist())
        d_pred.extend(dom_preds.tolist())

        file_labels.extend(batch["file_label"])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)
    d_true = np.array(d_true)
    d_pred = np.array(d_pred)

    class_acc = accuracy_score(y_true, y_pred)
    domain_acc = accuracy_score(d_true, d_pred)

    if len(np.unique(y_true)) == 2:
        class_auroc = roc_auc_score(y_true, y_prob)
    else:
        class_auroc = np.nan

    pred_control_pct = float(np.mean(y_pred == 0))
    pred_tamo_pct = float(np.mean(y_pred == 1))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "class_acc": class_acc,
        "class_auroc": class_auroc,
        "domain_acc": domain_acc,
        "pred_control_pct": pred_control_pct,
        "pred_tamo_pct": pred_tamo_pct,
        "confusion_matrix": cm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "d_true": d_true,
        "d_pred": d_pred,
        "file_labels": np.array(file_labels),
    }

## 7. Run one fold

In [ ]:
def run_fold(
    fold_config: dict,
    timepoint: int,
    n_epochs: int = N_EPOCHS,
    batch_size: int = BATCH_SIZE,
    lr: float = LR,
    domain_weight: float = 1.0,
):
    fold_id = fold_config["fold"]
    train_files = fold_config["train_files"]
    test_files = fold_config["test_files"]

    print(f"\n=== Fold {fold_id} ===")
    print("Train files:", train_files)
    print("Test files:", test_files)

    train_ds = build_dataset(train_files, timepoint=timepoint)
    test_ds = build_dataset(test_files, timepoint=timepoint)

    print("\nTrain summary:")
    display(summarise_dataset(train_ds))
    print("\nTest summary:")
    display(summarise_dataset(test_ds))

    train_sampler = make_weighted_sampler(train_ds)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    n_domains = len(set([f["domain_label"] for f in FILES]))
    model = DANN(
        n_channels=N_CHANNELS,
        n_classes=2,
        n_domains=n_domains,
        embedding_dim=256,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    for epoch in range(n_epochs):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            epoch=epoch,
            n_epochs=n_epochs,
            domain_weight=domain_weight,
        )
        eval_metrics = evaluate(model, test_loader)

        row = {
            "fold": fold_id,
            "epoch": epoch,
            **train_metrics,
            "test_class_acc": eval_metrics["class_acc"],
            "test_class_auroc": eval_metrics["class_auroc"],
            "test_domain_acc": eval_metrics["domain_acc"],
            "pred_control_pct": eval_metrics["pred_control_pct"],
            "pred_tamo_pct": eval_metrics["pred_tamo_pct"],
        }
        history.append(row)

        print(
            f"Epoch {epoch+1:03d}/{n_epochs} | "
            f"loss={row['loss']:.4f} | "
            f"class_acc={row['test_class_acc']:.3f} | "
            f"auroc={row['test_class_auroc']:.3f} | "
            f"domain_acc={row['test_domain_acc']:.3f} | "
            f"pred_control={row['pred_control_pct']:.2f}"
        )

    history_df = pd.DataFrame(history)
    final_eval = evaluate(model, test_loader)

    final_row = {
        "fold": fold_id,
        "timepoint": timepoint,
        "train_files": ", ".join(train_files),
        "test_files": ", ".join(test_files),
        "train_cells": len(train_ds),
        "test_cells": len(test_ds),
        "class_acc": final_eval["class_acc"],
        "class_auroc": final_eval["class_auroc"],
        "domain_acc": final_eval["domain_acc"],
        "pred_control_pct": final_eval["pred_control_pct"],
        "pred_tamo_pct": final_eval["pred_tamo_pct"],
        "tn": final_eval["confusion_matrix"][0, 0],
        "fp": final_eval["confusion_matrix"][0, 1],
        "fn": final_eval["confusion_matrix"][1, 0],
        "tp": final_eval["confusion_matrix"][1, 1],
    }

    return model, history_df, final_row, final_eval

## 8. Run DANN experiment at a fixed timepoint

The intended timepoint is usually `t=14`, because it is a later static snapshot while still being available in the original 15-timepoint files.

In [ ]:
all_histories = []
all_results = []
all_evals = {}

for fold_config in FOLDS:
    model, history_df, final_row, final_eval = run_fold(
        fold_config,
        timepoint=TIMEPOINT,
        n_epochs=N_EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        domain_weight=1.0,
    )

    all_histories.append(history_df)
    all_results.append(final_row)
    all_evals[fold_config["fold"]] = final_eval

history_all = pd.concat(all_histories, ignore_index=True)
results_df = pd.DataFrame(all_results)

history_all.to_csv(RESULTS_DIR / f"dann_history_t{TIMEPOINT}.csv", index=False)
results_df.to_csv(RESULTS_DIR / f"dann_results_t{TIMEPOINT}.csv", index=False)

results_df

## 9. Summary table

In [ ]:
summary = {
    "timepoint": TIMEPOINT,
    "n_folds": len(results_df),
    "class_acc_mean": results_df["class_acc"].mean(),
    "class_acc_std": results_df["class_acc"].std(ddof=1),
    "class_auroc_mean": results_df["class_auroc"].mean(),
    "class_auroc_std": results_df["class_auroc"].std(ddof=1),
    "domain_acc_mean": results_df["domain_acc"].mean(),
    "domain_acc_std": results_df["domain_acc"].std(ddof=1),
    "pred_control_pct_mean": results_df["pred_control_pct"].mean(),
    "pred_tamo_pct_mean": results_df["pred_tamo_pct"].mean(),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(RESULTS_DIR / f"dann_summary_t{TIMEPOINT}.csv", index=False)

summary_df

## 10. Export LaTeX tables

In [ ]:
latex_fold_table = results_df[
    [
        "fold",
        "class_acc",
        "class_auroc",
        "domain_acc",
        "train_cells",
        "test_cells",
        "pred_control_pct",
        "pred_tamo_pct",
    ]
].copy()

latex_fold_table["class_acc"] = latex_fold_table["class_acc"].map(lambda x: f"{x:.3f}")
latex_fold_table["class_auroc"] = latex_fold_table["class_auroc"].map(lambda x: f"{x:.3f}" if pd.notnull(x) else "N/A")
latex_fold_table["domain_acc"] = latex_fold_table["domain_acc"].map(lambda x: f"{x:.3f}")
latex_fold_table["pred_control_pct"] = latex_fold_table["pred_control_pct"].map(lambda x: f"{100*x:.1f}\\%")
latex_fold_table["pred_tamo_pct"] = latex_fold_table["pred_tamo_pct"].map(lambda x: f"{100*x:.1f}\\%")

latex_str = latex_fold_table.to_latex(
    index=False,
    escape=False,
    caption=f"DANN performance at timepoint {TIMEPOINT} using file-level grouped validation.",
    label=f"tab:dann_t{TIMEPOINT}_results",
)

with open(RESULTS_DIR / f"dann_results_t{TIMEPOINT}.tex", "w") as f:
    f.write(latex_str)

print(latex_str)

## 11. Mock results for writing while the experiment runs

Use this only as a placeholder. Replace these values with the real results after running the notebook.

In [ ]:
mock_results = pd.DataFrame([
    {
        "fold": 1,
        "class_acc": 0.581,
        "class_auroc": 0.612,
        "domain_acc": 0.581,
        "train_cells": 1557,
        "test_cells": 731,
        "pred_control_pct": 0.72,
        "pred_tamo_pct": 0.28,
    },
    {
        "fold": 2,
        "class_acc": 0.099,
        "class_auroc": 0.210,
        "domain_acc": 0.000,
        "train_cells": 1650,
        "test_cells": 638,
        "pred_control_pct": 0.91,
        "pred_tamo_pct": 0.09,
    },
    {
        "fold": 3,
        "class_acc": 0.723,
        "class_auroc": 0.704,
        "domain_acc": 0.000,
        "train_cells": 1769,
        "test_cells": 519,
        "pred_control_pct": 0.34,
        "pred_tamo_pct": 0.66,
    },
])

mock_summary = pd.DataFrame([{
    "class_acc_mean": mock_results["class_acc"].mean(),
    "class_acc_std": mock_results["class_acc"].std(ddof=1),
    "class_auroc_mean": mock_results["class_auroc"].mean(),
    "class_auroc_std": mock_results["class_auroc"].std(ddof=1),
    "domain_acc_mean": mock_results["domain_acc"].mean(),
    "domain_acc_std": mock_results["domain_acc"].std(ddof=1),
}])

mock_results.to_csv(RESULTS_DIR / "dann_mock_results.csv", index=False)
mock_summary.to_csv(RESULTS_DIR / "dann_mock_summary.csv", index=False)